# Forecasting des ventes — Baseline et modèle A

Ce notebook construit une prévision des ventes à **deux semaines d’horizon** et compare :
1. un **baseline naïf** : la prévision à deux semaines est égale aux ventes de la semaine courante (`sales_t`) ;
2. le **modèle A** : un modèle XGBoost utilisant `sales_t`, `lag_1`, `lag_2` et `lag_4`.

Les deux méthodes seront évaluées sur **le même jeu de test**, avec les mêmes métriques.

**Règle anti-fuite de données :** `target_sales` est la valeur à prédire. Elle ne doit jamais faire partie de `X_train_a` ni de `X_test_a`.

## 1. Charger et préparer les données

In [ ]:
from moobs.data import load_data, prepare_data

df = load_data()
df = prepare_data(df)

print("Dimensions :", df.shape)
print("Nombre de magasins :", df["store_id"].nunique())
print("Période :", df["date"].min(), "à", df["date"].max())

display(df.head())

## 2. Vérifier les données préparées

In [ ]:
print("Dimensions :", df.shape)
df.info()

print("\\nValeurs manquantes par colonne :")
print(df.isna().sum())

print(
    "\\nNombre de doublons magasin-date :",
    df.duplicated(subset=["store_id", "date"]).sum()
)

print("\\nDonnées triées par magasin et date :")
print(
    df.equals(
        df.sort_values(["store_id", "date"]).reset_index(drop=True)
    )
)

## 3. Créer les variables de prévision


In [ ]:
from moobs.features import create_features

forecast_df = create_features(df, horizon=2)

print("Dimensions après création des variables :", forecast_df.shape)
display(
    forecast_df[
        [
            "store_id", "date", "sales_t", "target_date",
            "target_sales", "lag_1", "lag_2", "lag_4",
            "lag_26", "lag_52", "rolling_mean_4"
        ]
    ].head(10)
)

`target_sales` correspond aux ventes à prédire deux semaines après l’observation courante. `sales_t` est la vente connue à la date d’origine. Le baseline utilisera uniquement `sales_t`.


## 4. Examiner les valeurs manquantes créées par les variables


In [ ]:
columns_to_check = [
    "target_date", "target_sales",
    "lag_1", "lag_2", "lag_4", "lag_26", "lag_52",
    "rolling_mean_4", "rolling_mean_8", "rolling_std_4",
]

print(forecast_df[columns_to_check].isna().sum())

## 6. Préparer le jeu de données avec le pipeline

`dataset.py` centralise le filtrage des lignes utilisables, le découpage temporel et la construction de `X` et `y`. Le notebook choisit seulement les variables de l’expérience et la date de coupure.

La cible `target_sales` est transmise séparément à `y` et la fonction refuse qu’elle figure dans la liste des variables explicatives.


In [ ]:
from moobs.dataset import build_dataset, common_test_index
from moobs.data import PROJECT_ROOT
import yaml

features_a = ["sales_t", "lag_1", "lag_2", "lag_4"]
target = "target_sales"

config_path = PROJECT_ROOT / "configs" / "baseline.yaml"
with open(config_path, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

cutoff_date = config["split"]["target_cutoff"]

dataset_a = build_dataset(
    forecast_df,
    features=features_a,
    cutoff_date=cutoff_date,
    target=target,
)

X_train_a = dataset_a.X_train
y_train_a = dataset_a.y_train
X_test_a = dataset_a.X_test
y_test_a = dataset_a.y_test

print("Date de coupure :", cutoff_date)
print("Lignes utilisables :", len(dataset_a.train_df) + len(dataset_a.test_df))
print("Lignes d'entraînement :", len(X_train_a))
print("Lignes de test :", len(X_test_a))
print("Période cible entraînement :", dataset_a.train_df["target_date"].min(), "à", dataset_a.train_df["target_date"].max())
print("Période cible test :", dataset_a.test_df["target_date"].min(), "à", dataset_a.test_df["target_date"].max())

assert target not in X_train_a.columns
assert target not in X_test_a.columns
assert not X_train_a.isna().any().any() and not y_train_a.isna().any()
assert not X_test_a.isna().any().any() and not y_test_a.isna().any()


## 7. Évaluer le baseline — dernière vente connue


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

y_pred_baseline = X_test_a["sales_t"].copy()

def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    # Le MAPE n'est défini que lorsque la valeur réelle n'est pas nulle.
    nonzero_mask = y_true != 0
    if nonzero_mask.any():
        mape = (
            np.abs((y_true[nonzero_mask] - y_pred[nonzero_mask])
                   / y_true[nonzero_mask]).mean() * 100
        )
    else:
        mape = np.nan

    return {"MAE": mae, "RMSE": rmse, "MAPE (%)": mape}

baseline_metrics = calculate_metrics(y_test_a, y_pred_baseline)

print("Résultats du baseline :")
for metric, value in baseline_metrics.items():
    print(f"{metric} : {value:.3f}")

## 8. Entraîner le modèle A avec XGBoost


In [ ]:
from xgboost import XGBRegressor

model_a = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model_a.fit(X_train_a, y_train_a)

print("Modèle A entraîné.")

## 9. Prédire et évaluer le modèle A


In [ ]:
y_pred_a = model_a.predict(X_test_a)

model_a_metrics = calculate_metrics(y_test_a, y_pred_a)

print("Résultats du modèle A :")
for metric, value in model_a_metrics.items():
    print(f"{metric} : {value:.3f}")

## 10. Comparer le baseline et le modèle A


In [ ]:
comparison_df = pd.DataFrame(
    [baseline_metrics, model_a_metrics],
    index=["Baseline — sales_t", "Modèle A — XGBoost"]
)

display(comparison_df)

## 11. Examiner l’importance des variables du modèle A


In [ ]:
feature_importance = (
    pd.Series(model_a.feature_importances_, index=features_a)
      .sort_values(ascending=False)
)

display(feature_importance.to_frame("importance"))

## 12. Préparer une comparaison future entre plusieurs jeux de variables

Si un modèle futur ajoute `lag_52`, il faut comparer les modèles sur les mêmes lignes de test. `common_test_index` calcule l'intersection des lignes de test utilisables par tous les jeux de variables. Cette cellule prépare les jeux A et B pour une comparaison ultérieure ; elle n'entraîne pas le modèle B.

In [ ]:
features_b = features_a + ["lag_52"]

shared_test_index = common_test_index(
    forecast_df,
    feature_sets=[features_a, features_b],
    cutoff_date=cutoff_date,
    target=target,
)

dataset_a_common = build_dataset(
    forecast_df, features_a, cutoff_date, target=target,
    test_index=shared_test_index,
)
dataset_b_common = build_dataset(
    forecast_df, features_b, cutoff_date, target=target,
    test_index=shared_test_index,
)

assert dataset_a_common.X_test.index.equals(dataset_b_common.X_test.index)
assert dataset_a_common.y_test.equals(dataset_b_common.y_test)

print("Test commun A/B :", len(shared_test_index), "lignes")
print("Entraînement A :", len(dataset_a_common.X_train), "lignes")
print("Entraînement B avec lag_52 :", len(dataset_b_common.X_train), "lignes")


In [2]:
!python -m pip install pytest


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [4]:
!python -m pytest --version

pytest 9.1.1


In [1]:

import pandas as pd
from moobs.predict import predict_from_dataframe

# Valeurs fictives : uniquement pour tester le fonctionnement technique.
test_df = pd.DataFrame({
    "sales_t": [100.0, 120.0],
    "lag_1": [95.0, 110.0],
    "lag_2": [90.0, 105.0],
    "lag_4": [85.0, 100.0],
    "rolling_mean_4": [92.0, 108.0],
    "rolling_mean_8": [88.0, 103.0],
    "rolling_std_4": [5.0, 6.0],
})

predictions = predict_from_dataframe(test_df, horizon=1)

print(predictions[["predicted_sales"]])


   predicted_sales
0       107.496048
1       114.016708


In [2]:
from pathlib import Path

path = Path("reports/backtesting_results.csv")

print("Fichier trouvé :", path.exists())

if path.exists():
    import pandas as pd
    results = pd.read_csv(path)
    print(results.head())
    print("\nColonnes :", results.columns.tolist())

Fichier trouvé : False


In [3]:
pip install reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 13.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 14.9 MB/s  0:00:006.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [reportlab]━━━━━━━━ 2/3 [reportlab]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install reportlab matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 15.1 MB/s  0:00:00a 0:00:01m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 13.9 MB/s  0:00:006.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 11.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [matplotlib] 5/6 [matplotlib]solver]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
